In [1]:
# Установка Demucs из PyPI
%pip install demucs
%pip install vosk
%pip install soundfile faster_whisper
%pip install hf_xet

# Установка остальных библиотек
%pip install noisereduce pydub librosa soundfile openai-whisper mir_eval vock faster_whisper
%pip install torch torchaudio numpy

# Установка PyTorch с поддержкой CUDA 12.1 (если нужна GPU)
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not find a version that satisfies the requirement vock (from versions: none)

[notice] A new release 

In [2]:
##%% Импорты
import soundfile as sf
import noisereduce as nr
from pydub import AudioSegment
from pydub.effects import normalize
import whisper
from demucs.apply import apply_model
from demucs.pretrained import get_model
from demucs.audio import AudioFile
import tempfile
import numpy as np
import librosa
from mir_eval.separation import bss_eval_sources
import torch
from vosk import Model, KaldiRecognizer
import wave
import json
from faster_whisper import WhisperModel

##%% Проверка GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [3]:
##%% 0.
def channel_dubl(input_path, output_path="converted.wav"):
    sound = AudioSegment.from_file(input_path)  # или .wav
    sound = sound.set_channels(2).set_frame_rate(44100)
    sound.export(output_path, format="wav")
    return output_path


##%% 1. Выделение вокала с помощью Demucs
def separate_vocals(input_path, output_path="vocals.wav"):
    model = get_model(name="htdemucs").to(device)
    with tempfile.TemporaryDirectory() as tmpdir:
        wav = AudioFile(input_path).read(streams=0, samplerate=44100)
        sources = apply_model(model, wav[None], device=device, split=True, progress=True)
        vocals = sources[0][3]  # [vocals, drums, bass, other]
        sf.write(output_path, vocals.T, 44100)
    return output_path


def enhance_audio(input_path, output_path="enhanced.wav", sample_rate=44100, chunk_duration=10):
    """Убирает шум и усиливает вокал по частям."""
    audio, sr = sf.read(input_path)

    # Если стерео → в моно
    if len(audio.shape) > 1:
        audio = np.mean(audio, axis=1)

    chunk_size = chunk_duration * sample_rate
    enhanced_chunks = []

    for i in range(0, len(audio), chunk_size):
        chunk = audio[i:i + chunk_size]
        try:
            clean_chunk = nr.reduce_noise(y=chunk, sr=sr, prop_decrease=0.9)
        except Exception as e:
            print(f"⚠️ Ошибка в чанке {i // chunk_size}: {e}")
            clean_chunk = chunk  # fallback: оставить как есть
        enhanced_chunks.append(clean_chunk)

    enhanced_audio = np.concatenate(enhanced_chunks)
    sf.write(output_path, enhanced_audio, sample_rate)
    return output_path


##%% 3. Подавление шума и улучшение вокала
def postprocess_audio(input_path, output_path="vocal_boosted.wav"):
    audio = AudioSegment.from_file(input_path)
    audio = normalize(audio)
    audio = audio.high_pass_filter(100)
    audio = audio.low_pass_filter(10000)
    audio += 4
    audio = audio.compress_dynamic_range(threshold=-20.0, ratio=5.0)
    audio.export(output_path, format="wav")
    return output_path


def transcribe_audio(path):
    model = whisper.load_model("large", device=device)
    return model.transcribe(
        path,
        language="ru",
        condition_on_previous_text=False,
        word_timestamps=True,
        # vad_filter=True,  # убирает молчание
    )["text"]


def transcribe_with_vosk(audio_path, model_path='models/vosk-model-ru-0.42'):
    import torchaudio

    def convert_to_mono_16k(input_path, output_path="converted.wav"):
        waveform, sr = torchaudio.load(input_path)
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
        mono = waveform.mean(dim=0, keepdim=True)
        resampled = resampler(mono)
        torchaudio.save(output_path, resampled, 16000)
        return output_path

    # Открываем WAV файл
    wf = wave.open(audio_path, "rb")
    # if wf.getnchannels() != 1 or wf.getsampwidth() != 2 or wf.getframerate() != 16000:
    #     audio_path = convert_to_mono_16k(audio_path)
    #     wf = wave.open(audio_path, "rb")

    # raise ValueError("Файл должен быть WAV 16кГц, 16bit, моно")

    model = Model(model_path)
    rec = KaldiRecognizer(model, wf.getframerate())

    results = []
    while True:
        data = wf.readframes(4000)
        if len(data) == 0:
            break
        if rec.AcceptWaveform(data):
            result = json.loads(rec.Result())
            results.append(result.get("text", ""))

    final_result = json.loads(rec.FinalResult())
    results.append(final_result.get("text", ""))

    return " ".join(results)


def transcribe_faster_whisper(audio_path):
    model_size = "large-v2"

    # Использовать GPU, если есть (можно указать "cpu")
    model = WhisperModel(model_size, device=device)
    print("🎙️ Распознавание песни...")

    segments, info = model.transcribe(audio_path, language="ru")

    full_text = ""
    for segment in segments:
        full_text += segment.text.strip() + " "

    return full_text.strip()

In [4]:
def test_vac(reference_vocals, vocals):
    # Загрузка исходного вокала и предсказанного
    print(f"Тестирование {reference_vocals} {vocals}")
    true_vocals, _ = librosa.load(reference_vocals, sr=44100, mono=True)
    predicted_vocals, _ = librosa.load(vocals, sr=44100, mono=True)

    # Подгонка длины
    min_len = min(len(true_vocals), len(predicted_vocals))
    true_vocals = true_vocals[:min_len]
    predicted_vocals = predicted_vocals[:min_len]

    # BSS Eval метрики
    sdr, sir, sar, _ = bss_eval_sources(np.expand_dims(true_vocals, axis=0),
                                        np.expand_dims(predicted_vocals, axis=0))

    print(f"SDR: {sdr[0]:.2f} dB")
    print(f"SIR: {sir[0]:.2f} dB")
    print(f"SAR: {sar[0]:.2f} dB")


In [6]:

##%% 4. Полный pipeline
def full_pipeline(audio_path):
    print("🎙️ Выделение вокала...")
    audio_path  = channel_dubl(audio_path)
    vocals_path = separate_vocals(audio_path)  # Demucs
    test_vac(vocals_path, audio_path)
    print("Деление")
    enhanced_path = enhance_audio(vocals_path)
    print("🧼 Подавление шума и фильтрация...")  # noisereduce
    final_path = postprocess_audio(enhanced_path)  # pyd
    test_vac(final_path, vocals_path)
    print(enhanced_path, vocals_path, final_path)
    final_path = 'vocal_boosted.wav'
    print("📝 Транскрипция...")
    text = transcribe_faster_whisper(final_path)  # Whisper

    return text


##%% Пример использования
result = full_pipeline("src/Баксанская.wav")
print(result)
# result = full_pipeline("src/Боевая-пехотная.wav")
# print(result)


📝 Транскрипция...


[2025-05-19 13:51:05.172] [ctranslate2] [thread 10458] [warning] The compute type inferred from the saved model is float16, but the target device or backend do not support efficient float16 computation. The model weights have been automatically converted to use the float32 compute type instead.


🎙️ Распознавание песни...
Где снега тропинки заметают, Где лавины грозные шумят, Эту песню сложил и распевает Альпинистов боевой отряд. Нам в боях родными стали горы, Не страшны бураны и пурга. Там приказ, недолгие были сборы На разведку в логово врага. Помнишь, товарищ, белые снега, Стройный лес боксана, блин, даже врага. Помнишь, гранату и записку в ней На скалистом гребне для грядущих дней. Помнишь, товарищ, вой ночной пурги, Помнишь, как бежали в панике враги, Как загрохотал твой грозный автомат. Помнишь, как вернулись мы с тобой в отряд, На костре в дыму трещали ветки, В котелке дымился крепкий чай. Ты пришел усталый из разведки, Много пил и столько же молчал. Синими замерзшими руками Протирал вспотевший автомат. И о чем ты думал временами, Головой откинувшись назад. Помнишь, товарищ, белые снега, Стройный лес боксана, блин, даже врага. Помнишь, гранату и записку в ней На скалистом гребне для грядущих дней. Помнишь, товарищ, вой ночной пурги, Помнишь, как бежали в панике враги, Ка

In [95]:
import subprocess

# Путь к твоему скрипту .sh
script_path = "./hh.sh"

# Запускаем скрипт и ждём окончания
result = subprocess.run(["bash", script_path], capture_output=True, text=True)

# Выводим stdout и stderr для отладки
print("stdout:", result.stdout)
print("stderr:", result.stderr)

# Проверяем код возврата (0 — успех)
if result.returncode == 0:
    print("Скрипт выполнен успешно!")
else:
    print(f"Ошибка выполнения скрипта, код {result.returncode}")


stdout: 📦 Установка зависимостей...
Defaulting to user installation because normal site-packages is not writeable
⬇️ Скачивание модели Vosk для русского языка...
📦 Распаковка модели...
✅ Установка завершена. Модель находится в models/vosk-model-ru-0.42

stderr: 
[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
--2025-05-19 09:51:22--  https://alphacephei.com/vosk/models/vosk-model-ru-0.42.zip
Resolving alphacephei.com (alphacephei.com)... 188.40.21.16, 2a01:4f8:13a:279f::2
Connecting to alphacephei.com (alphacephei.com)|188.40.21.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1937602113 (1.8G) [application/zip]
Saving to: ‘vosk-model-ru-0.42.zip’

     0K .......... .......... .......... .......... ..........  0% 1.17M 26m24s
    50K .......... .......... .......... .......... ..........  0% 1.15M 26m36s
   100K .......... .......... .......... .......... ..........  0% 1.16M 26m36s
